# PEEL Phase 1 -- Semantic Analysis Pipeline

Stem extraction, GlossBERT word-sense disambiguation, and semantic
clustering. See the root [README.md](../README.md) for the full pipeline
overview, setup instructions, and the data-directory / decision-log
conventions used here.

Set `CORPUS_NAME` in the config cell below to the corpus you want to
analyze -- its raw text must exist at
`data/<CORPUS_NAME>/raw/<CORPUS_NAME>.txt`.

In [ ]:
# IMPORTS
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import torch
import spacy
import nltk
from nltk.stem import PorterStemmer

from common.paths import CorpusPaths, resources_dir
from common.decisions import DecisionLog
from phase1 import pipeline

# REQUIRED DOWNLOADS
nltk.download("wordnet")
nltk.download("omw-1.4")

In [ ]:
# DEVICE
# This pipeline requires a GPU or else it'll take a long time or not even run at all.
# If output is "cuda", proceed.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\nUsing device: {device}")

In [ ]:
# CONFIG

CORPUS_NAME = "Boisseau"  # only line to edit to switch corpus

paths = CorpusPaths(CORPUS_NAME)
paths.ensure_dirs()
decisions = DecisionLog(CORPUS_NAME, phase="phase1")

TOP_PERCENTILE = 0.50        # top 50% most frequent stems
MAX_STEMS = 150               # maximum number of stems to return
MAX_SENTENCES_PER_STEM = 5    # sentences collected per stem
MAX_SYNSETS = 5               # WordNet definitions considered per stem -- a ceiling, not a
                               # guarantee: capped further by how many senses WordNet actually
                               # has for a given word's part of speech (e.g. "call" as a verb
                               # has 28 senses, but "human" as an adjective has only 3)
MAX_CLUSTER_SIZE = 10         # cluster size above which it gets re-split
MIN_CLUSTERS = 5              # HDBSCAN min_cluster_size
MIN_CLUSTER_LEN = 3           # minimum stems for a rerun/noise cluster to be valid

GLOSSBERT_MODEL_ID = "jvomiranda/GlossBERT_Checkpoint"  # fetched from the Hugging Face Hub
LANG_MODEL = "en_core_web_sm"
SENTENCE_EMBEDDER = "all-MiniLM-L6-v2"

nlp = spacy.load(LANG_MODEL)
stemmer = PorterStemmer()

In [ ]:
# READ TEXT AND EXTRACT TOP STEMS

with open(paths.raw_txt(), "r", encoding="utf-8") as f:
    text = f.read()

doc = nlp(text)

top_stems = pipeline.extract_top_stems(doc, stemmer, TOP_PERCENTILE, MAX_STEMS)

print(f"\nSelected {len(top_stems)} top stems\n")
for stem, count in top_stems.items():
    print(f"{stem}: {count}")

with open(paths.top_stems(), "w", encoding="utf-8") as out:
    out.write("stem\tcount\n")
    for stem, count in top_stems.items():
        out.write(f"{stem}\t{count}\n")

print(f"\nSaved results to {paths.top_stems()}")

In [ ]:
# GLOSSBERT WORD-SENSE DISAMBIGUATION
# GlossBERT_Checkpoint is fetched from the Hugging Face Hub the first time this runs
# and cached locally afterwards -- no manual download/placement needed.

print("\nLoading GlossBERT checkpoint...")
tokenizer, model = pipeline.load_glossbert(GLOSSBERT_MODEL_ID, device)
print("GlossBERT loaded successfully.")

print("\nMapping stem occurrences...")
stem_occurrences = pipeline.map_stems_to_sentences(doc, top_stems, stemmer)

print("\nRunning GlossBERT analysis...")
accepted_definitions, flagged_words = pipeline.run_glossbert_analysis(
    top_stems, stem_occurrences, tokenizer, model, device,
    max_sentences_per_stem=MAX_SENTENCES_PER_STEM, max_synsets=MAX_SYNSETS,
)

pipeline.print_accepted_definitions(accepted_definitions)
pipeline.print_flagged_words(flagged_words)
print("\nDone.")

In [ ]:
# USER REVIEW OF FLAGGED TERMS
# Every choice made here is appended to data/<CORPUS_NAME>/decisions/phase1_decisions.jsonl
# via `decisions.record(...)` for later audit -- it does not change the interactive flow.

pipeline.run_flagged_term_review(flagged_words, accepted_definitions, decisions)

print("\nSaving updated results...")
pipeline.save_glossbert_output(accepted_definitions, paths.glossbert_output())
print(f"\nUpdated results saved to {paths.glossbert_output()}")

In [ ]:
# SEMANTIC EMBEDDINGS AND INITIAL CLUSTERING

embedder, stem_texts, stem_names, embeddings = pipeline.build_stem_embeddings(
    accepted_definitions, SENTENCE_EMBEDDER
)
labels, clusters = pipeline.cluster_stem_embeddings(
    embeddings, stem_names, MIN_CLUSTERS, min_cluster_len=MIN_CLUSTER_LEN,
)

print(f"Stems embedded: {len(stem_names)}")
print(f"Clusters found (excluding noise): {len(clusters)}")

In [ ]:
# CLUSTER NAMING PREVIEW

stem_word_frequencies = pipeline.build_stem_word_frequency_table(stem_occurrences)
renamed_clusters = pipeline.rename_clusters(clusters, stem_word_frequencies)
pipeline.print_named_clusters(renamed_clusters, "SEMANTIC CLUSTERS (BEFORE RECLUSTERING)")

In [ ]:
# RECLUSTER LARGE CLUSTERS
# Clusters bigger than MAX_CLUSTER_SIZE are re-embedded with richer context
# (observed words + sentences + candidate senses) and re-split.

clusters, large_subclusters = pipeline.recluster_large_clusters(
    clusters, stem_occurrences, embedder, tokenizer, model, device,
    max_cluster_size=MAX_CLUSTER_SIZE, min_clusters=MIN_CLUSTERS,
    min_cluster_len=MIN_CLUSTER_LEN, max_synsets=MAX_SYNSETS,
)

pipeline.print_raw_clusters(clusters, "UPDATED CLUSTERS")

In [ ]:
pipeline.print_raw_clusters(large_subclusters, "LARGE CLUSTER SUBCLUSTERS")

In [ ]:
# RERUN NOISE
# Stems HDBSCAN initially treated as noise may still form a valid, smaller cluster.

noise_stems = pipeline.extract_noise_stems(stem_names, labels)

noise_clusters = pipeline.recluster_noise(
    noise_stems, stem_occurrences, embedder, tokenizer, model, device,
    min_clusters=MIN_CLUSTERS, min_cluster_len=MIN_CLUSTER_LEN, max_synsets=MAX_SYNSETS,
)

pipeline.print_raw_clusters(noise_clusters, "NOISE CLUSTERS")

In [ ]:
# FINAL CLUSTER NAMING
# Renames both the (post-recluster) main clusters and the recovered noise
# clusters, merging them into one named set with collision-safe names.

stem_word_frequencies = pipeline.build_stem_word_frequency_table(stem_occurrences)

renamed_clusters = pipeline.rename_clusters(clusters, stem_word_frequencies)
renamed_noise_clusters = pipeline.rename_clusters(noise_clusters, stem_word_frequencies)
renamed_clusters = pipeline.merge_named_clusters(renamed_clusters, renamed_noise_clusters)

pipeline.print_named_clusters(renamed_clusters, "SEMANTIC CLUSTERS")

In [ ]:
# CLUSTER N-GRAMS
# Mines representative multi-word n-grams per cluster (frequent enough
# globally, and overlapping one of the cluster's own stems).

sentence_cache = pipeline.build_sentence_token_cache(nlp, accepted_definitions, stemmer, use_lemmas=True)
global_ngram_counter, percentile_threshold = pipeline.build_global_ngram_statistics(sentence_cache)

all_clusters = {**clusters, **noise_clusters}
cluster_ngrams = pipeline.extract_cluster_ngrams(
    all_clusters, accepted_definitions, sentence_cache,
    global_ngram_counter, percentile_threshold, stemmer, use_lemmas=True,
)

renamed_clusters = pipeline.attach_ngrams(renamed_clusters, cluster_ngrams)

In [ ]:
# INTERACTIVE CLUSTER REVIEW
# Every choice made here is appended to data/<CORPUS_NAME>/decisions/phase1_decisions.jsonl
# via `decisions.record(...)` for later audit -- it does not change the interactive flow.

final_clusters, excluded_cluster_stems, excluded_cluster_ngrams, all_original_stems = (
    pipeline.run_cluster_review(renamed_clusters, decisions)
)

In [ ]:
# VOYANT SETTINGS
# Every choice made here is appended to data/<CORPUS_NAME>/decisions/phase1_decisions.jsonl
# via `decisions.record(...)` for later audit -- it does not change the interactive flow.

corpus_id, smart_stopwords, use_smart_stopwords = pipeline.run_voyant_settings(
    decisions, resources_dir() / "stop.en.smart.txt"
)

In [ ]:
# SAVE PHASE 1 STATE JSON

phase1_state = pipeline.build_phase1_state(
    corpus_id=corpus_id,
    all_original_stems=all_original_stems,
    excluded_cluster_stems=excluded_cluster_stems,
    excluded_cluster_ngrams=excluded_cluster_ngrams,
    final_clusters=final_clusters,
    smart_stopwords=smart_stopwords,
    use_smart_stopwords=use_smart_stopwords,
)

pipeline.save_phase1_state(phase1_state, paths.phase1_state_json())
print(f"\nSaved JSON to {paths.phase1_state_json()}")

In [ ]:
# HTML CLUSTER EXPORT

html = pipeline.build_cluster_html(final_clusters, CORPUS_NAME)
pipeline.save_html(html, paths.phase1_html())
print(f"\nHTML cluster snippet written:\n{paths.phase1_html()}")